# 1. Basic Convolutional Autoencoder (CNN Autoencoder)
### Problem it solves
Used for basic image compression and feature extraction. Linear layers lose spatial information in images, so CNNs are better for vision tasks.

### Architecture / Parameter Changes
*   **Architecture**: Uses `Conv2D` for feature extraction and `MaxPooling2D` for compression (Encoder). Uses `Conv2D` and `UpSampling2D` for reconstruction (Decoder).
*   **Loss Function**: Usually Mean Squared Error (`mse`) since we are predicting pixel values.
*   **Activation**: `relu` for hidden layers to prevent vanishing gradients, `sigmoid` at the output layer to scale predictions securely between `[0, 1]`.



In [ ]:
import numpy as np                          # Import numpy for numerical operations
from tensorflow.keras.datasets import mnist # Import MNIST dataset (digits 0–9)
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D # Layers used in CNN autoencoder
from tensorflow.keras.models import Model   # Used to define the model
import matplotlib.pyplot as plt             # For visualization

# Load data
(x_train, _), (x_test, _) = mnist.load_data()  
# Loads MNIST dataset (x_train, x_test = images). Labels '_' are ignored because autoencoder is unsupervised.

# Normalize and reshape
x_train = x_train.astype('float32') / 255.  
# Convert pixel values (0–255) to float and normalize to range [0,1]

x_test = x_test.astype('float32') / 255.    
# Same normalization for test data

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))  
# Reshape training data to (num_samples, height, width, channels) - MNIST is grayscale → 1 channel

x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))    
# Same reshape for test data

# ================= ENCODER =================
input_img = Input(shape=(28,28,1))  
# Define input layer with image size 28x28x1

x = Conv2D(16, (3,3), activation='relu', padding='same')(input_img)  
# Apply convolution: 16 filters, kernel size 3x3, ReLU activation, padding='same' keeps output size same (28x28)

x = MaxPooling2D((2,2), padding='same')(x)   
# Downsample image by factor 2 → (28x28 → 14x14) - Keeps important features

x = Conv2D(32, (3,3), activation='relu', padding='same')(x)  
# Another convolution with 32 filters for deeper feature extraction

encoded = MaxPooling2D((2,2), padding='same')(x)  
# Further downsampling → (14x14 → 7x7) - This is the compressed representation (latent space)

# ================= DECODER =================
x = Conv2D(32, (3,3), activation='relu', padding='same')(encoded)  
# Start decoding: extract features again from compressed data

x = UpSampling2D((2,2))(x)   
# Upsample → (7x7 → 14x14) - Reverse of pooling

x = Conv2D(16, (3,3), activation='relu', padding='same')(x)  
# Reduce channels back while reconstructing

x = UpSampling2D((2,2))(x)   
# Upsample again → (14x14 → 28x28)

decoded = Conv2D(1, (3,3), activation='sigmoid', padding='same')(x)  
# Final output layer: 1 channel (grayscale), sigmoid → output in range [0,1] (same as input)

# ================= MODEL =================
autoencoder = Model(input_img, decoded)  
# Define model: input → reconstructed output

autoencoder.compile(optimizer='adam', loss='mse')  
# Compile model: optimizer='adam' → adjusts weights, loss='mse' → measures reconstruction error

# Train
autoencoder.fit(x_train, x_train, epochs=5, batch_size=128, shuffle=True, validation_data=(x_test, x_test))
# Train model: input = x_train, output = x_train (reconstruct)

# Reconstruction & Visualization
decoded_imgs = autoencoder.predict(x_test)  
n = 5  
for i in range(n):
    plt.subplot(2,n,i+1)
    plt.imshow(x_test[i].reshape(28,28), cmap='gray')
    plt.axis('off')
    plt.subplot(2,n,i+n+1)
    plt.imshow(decoded_imgs[i].reshape(28,28), cmap='gray')
    plt.axis('off')
plt.show()  



# 2. Denoising Autoencoder (Noise Removal)
### Problem it solves
Images in real life are often noisy (sensor noise, blur, compression artifacts). Model learns to map a corrupted/noisy signal to the original uncorrupted signal.

### Architecture / Parameter Changes
*   **Architecture**: Similar to basic AE, but often more filters/layers are used to capture richer reconstruction semantics.
*   **Data Preparation**: We manually inject noise (e.g. Gaussian noise) to the inputs. 
*   **Input vs Output**: During training, `X_train` is the NOISY image, while `Y_train` (target) is the CLEAN image.
*   **Loss Function**: Still `mse` (Mean Squared Error) because we measure difference between predicted clean image and ground-truth clean image. For specific human-vision improvements, perceptual losses could be used instead.



In [ ]:
# [Assuming x_train and x_test are already loaded and scaled]
# We will regenerate a new instance model for Denoising!

import numpy as np  
# Used for numerical operations and generating random noise
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D  
from tensorflow.keras.models import Model  

# ================= ADDING NOISE TO DATA =================
noise_factor = 0.5  
# Controls amount of noise (higher = more noise)

x_train_noisy = x_train + noise_factor * np.random.normal(size=x_train.shape)  
# Add Gaussian noise to training images

x_test_noisy = x_test + noise_factor * np.random.normal(size=x_test.shape)  
# Add Gaussian noise to testing images

# Clip values to [0,1]
x_train_noisy = np.clip(x_train_noisy, 0, 1)  
# Ensures pixel values remain valid (between 0 and 1) after noise addition
x_test_noisy = np.clip(x_test_noisy, 0, 1)  

# ================= ENCODER =================
input_img = Input(shape=(28,28,1))  
# Input layer: image size

x = Conv2D(32,(3,3),activation='relu',padding='same')(input_img)  
# 32 filters → detect edges, patterns. kernel = 3x3. ReLU → non-linearity. padding='same' → output size remains 28x28

x = MaxPooling2D((2,2),padding='same')(x)  
# Downsample → (28x28 → 14x14) - Reduces spatial size → compression

x = Conv2D(64,(3,3),activation='relu',padding='same')(x)  
# Increase filters → learn more complex features

encoded = MaxPooling2D((2,2),padding='same')(x)  
# Further compression → (14x14 → 7x7) - This is latent representation (compressed form)

# ================= DECODER =================
x = Conv2D(64,(3,3),activation='relu',padding='same')(encoded)  
# Start reconstruction from compressed features

x = UpSampling2D((2,2))(x)  
# Upsample → (7x7 → 14x14)

x = Conv2D(32,(3,3),activation='relu',padding='same')(x)  
# Reduce feature depth

x = UpSampling2D((2,2))(x)  
# Upsample → (14x14 → 28x28)

decoded = Conv2D(1,(3,3),activation='sigmoid',padding='same')(x)  
# Final output layer: 1 channel image, sigmoid → ensures output in [0,1]

# ================= MODEL =================
denoising_ae = Model(input_img, decoded)  
# Defines mapping: noisy image → clean image

denoising_ae.compile(optimizer='adam', loss='mse')  
# optimizer='adam' → adaptive learning (fast convergence), loss='mse' → reconstruction error

# ================= TRAIN =================
denoising_ae.fit(x_train_noisy, x_train, epochs=5, batch_size=128, validation_data=(x_test_noisy, x_test))  
# CRITICAL DIFFERENCE: Input = noisy image (`x_train_noisy`), Output / Target = clean image (`x_train`). Model learns to remove noise!



# 3. Sparse Autoencoder (Feature Selection)
### Problem it solves
A regular autoencoder might simply learn the "identity function" (just passing inputs strictly to outputs) without extracting meaningful features, especially if the hidden layer represents more dimensions than the input data. Sparse Autoencoder solves this by enforcing *sparsity*—meaning only a few neurons can fire at once.

### Architecture / Parameter Changes
*   **Architecture Constraint**: Addition of an L1 regularization term directly on the node activations of the hidden bottleneck layer.
*   **Sparsity Penalty Parameter**: Applied via `activity_regularizer=regularizers.l1(...)`. This mathematical penalty forces most neuron activations to hit 0, except for those that detect crucial robust features.



In [ ]:
from tensorflow.keras import regularizers
# Import regularizers to apply mathematical L1 sparsity constraints

input_img = Input(shape=(28,28,1))
# Define input image shape

# ================= ENCODER WITH SPARSITY =================
x = Conv2D(16,(3,3),activation='relu',padding='same')(input_img)
x = MaxPooling2D((2,2),padding='same')(x)
# First downsample

encoded = Conv2D(8,(3,3),activation='relu',padding='same', activity_regularizer=regularizers.l1(1e-5))(x)
# Deepest layer of the encoder (latent space). 
# Notice `activity_regularizer=regularizers.l1(1e-5)`: This L1 penalty forces many activations to 0.
# It makes the network ignore trivial details and learn highly robust patterns.

# ================= DECODER =================
x = Conv2D(8,(3,3),activation='relu',padding='same')(encoded)
# Start of decoder

x = UpSampling2D((2,2))(x)
# Upsample back to 28x28

decoded = Conv2D(1,(3,3),activation='sigmoid',padding='same')(x)
# Reconstruct original image

# ================= MODEL =================
sparse_ae = Model(input_img, decoded)
sparse_ae.compile(optimizer='adam', loss='mse')
# Compile model with regular Mean Squared Error loss

# Train the Sparse Autoencoder
sparse_ae.fit(x_train, x_train, epochs=5, batch_size=128, validation_data=(x_test, x_test))
# Learning to reconstruct input, but strongly constrained by the L1 sparsity penalty inside!



# 4. Deep Autoencoder
### Problem it solves
Simple shallow autoencoders fail to learn highly abstract, compositional patterns within data. Deep Autoencoders solve this by stacking multiple hidden layers, allowing the network to assemble basic feature edges into more complex features (like circles, eyes, digits).

### Architecture / Parameter Changes
*   **Architecture Changes**: We heavily deepen the stack, adding successive `Conv2D` & `MaxPooling2D` blocks. Feature map channels incrementally increase (32 -> 64 -> 128). The representation scales down spatially, but grows extremely dense logically.



In [ ]:
# ================= DEEP ENCODER =================
input_img = Input(shape=(28,28,1))
# Input definition

x = Conv2D(32,(3,3),activation='relu',padding='same')(input_img)
x = MaxPooling2D((2,2),padding='same')(x)
# First compression step

x = Conv2D(64,(3,3),activation='relu',padding='same')(x)
x = MaxPooling2D((2,2),padding='same')(x)
# Second compression step

x = Conv2D(128,(3,3),activation='relu',padding='same')(x)
encoded = MaxPooling2D((2,2),padding='same')(x)
# Deepest latent representation. 
# Rich channel depth (128) means we've heavily abstracted structural features.

# ================= DEEP DECODER =================
x = Conv2D(128,(3,3),activation='relu',padding='same')(encoded)
x = UpSampling2D((2,2))(x)
# First reconstruction step

x = Conv2D(64,(3,3),activation='relu',padding='same')(x)
x = UpSampling2D((2,2))(x)
# Second reconstruction step

x = Conv2D(32,(3,3),activation='relu',padding='same')(x)
x = UpSampling2D((2,2))(x)
# Final upsampling brings spatial dimension back to original 28x28

decoded = Conv2D(1,(3,3),activation='sigmoid',padding='same')(x)
# Output

# ================= MODEL =================
deep_ae = Model(input_img, decoded)
deep_ae.compile(optimizer='adam', loss='mse')
# Compiling Deep Autoencoder with standard MSE loss

deep_ae.fit(x_train, x_train, epochs=5, batch_size=128, validation_data=(x_test, x_test))
# Will typically yield much lower training error than a shallow AE owing to extreme capacity



# 5. Variational Autoencoder (VAE) (Generative Model)
### Problem it solves
A regular autoencoder creates a spotty, discrete latent representation. If we pick a random point in that space, it will likely decode into mere static/garbage. VAEs enforce a continuous, perfectly arranged underlying distribution space. Consequently, VAEs can *generate entirely new, fake realistic data* by sampling variables randomly exactly from that distribution space.

### Architecture / Parameter Changes
*   **Latent Distribution Paradigm**: The encoder produces a `Mean` ($\mu$) and `Variance` ($\Sigma$) vector defining a probabilistic Gaussian space, rather than a single direct latent vector.
*   **Reparameterization Trick**: We sample a mathematical point from that specific normal distribution smoothly mapping it via `Lambda(sampling)` structure, guaranteeing clean backpropagation.
*   **Loss Function**: Fundamentally altered. Total Loss = `Reconstruction Loss` (Binary Crossentropy) + `KL Divergence`. The KL Divergence regularizes the model by enforcing the distribution tightly resembles a canonical standard Normal bell curve distribution $N(0, 1)$.



In [ ]:
from tensorflow.keras.layers import Lambda, Flatten, Dense, Reshape
import tensorflow.keras.backend as K
import tensorflow as tf

latent_dim = 2  
# We are shrinking images into a simple 2-Dimensional continuous distribution space (for easy generation)

input_img = Input(shape=(28,28,1))
x = Flatten()(input_img)
# Flatten 2D image data into 1D for Dense layers processing
x = Dense(128, activation='relu')(x)
# Dense hidden layer mapping

# ================= DISTRIBUTION PARAMETERS =================
z_mean = Dense(latent_dim)(x)
# The network splits logic here mapping independent parameters: The MEAN of the distribution
z_log_var = Dense(latent_dim)(x)
# And the LOG VARIANCE of the distribution

# ================= REPARAMETERIZATION TRICK =================
def sampling(args):
    z_mean, z_log_var = args
    epsilon = K.random_normal(shape=(K.shape(z_mean)[0], latent_dim))
    # Epsilon introduces randomness (sampled from standard normal N(0, 1))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon
    # Returning the sampled point z. This trick ensures computational gradient tracks cleanly backwards!

z = Lambda(sampling)([z_mean, z_log_var])
# The Latent space Z is now properly dynamically sampled

# ================= DECODER =================
decoder_h = Dense(128, activation='relu')
# Decoding the 2D distribution point back upwards
decoder_mean = Dense(784, activation='sigmoid')
# Predicting 784 pixel continuous values

h_decoded = decoder_h(z)
x_decoded = decoder_mean(h_decoded)
decoded = Reshape((28,28,1))(x_decoded)
# Reshaping mapping from flat (784,) format back smoothly to 2D image framework (28x28x1)

# ================= MODEL & CUSTOM LOSS =================
vae = Model(input_img, decoded)

# VAE loss = Reconstruction Loss + KL Divergence 
reconstruction_loss = tf.keras.losses.binary_crossentropy(K.flatten(input_img), K.flatten(decoded))
reconstruction_loss *= 784 
# Scale reconstruction loss cleanly over all constituent pixels 

kl_loss = 1 + z_log_var - K.square(z_mean) - K.exp(z_log_var)
kl_loss = K.sum(kl_loss, axis=-1)
kl_loss *= -0.5
# KL divergence penalizes parameters straying too sharply away from standard Normal shape N(0, 1)

vae_loss = K.mean(reconstruction_loss + kl_loss)
# Aggregate the losses
vae.add_loss(vae_loss)
# Improvise insertion of Custom Total Logic loss directly internally into model tracker

vae.compile(optimizer='adam')
# Notice we purposefully do not pass `loss=''` explicit argument here inside compile since we utilized native `add_loss`

# Train the VAE
vae.fit(x_train, epochs=5, batch_size=128, validation_data=(x_test, None))
# Output dataset argument natively skipped because internal Custom tracking directly evaluates `input_img` internally!



# 6. Contractive Autoencoder (Robust feature learning)
### Problem it solves
Highly similar to a Denoising Autoencoder structurally but tackles it mathematically. Instead of artificially altering the data itself, we severely penalize the mapping functions internal sensitivity. We desire the internal features mapped inside the tight hidden layer to be INVARIANT (impervious) to minuscule changes to the input parameters.

### Architecture / Parameter Changes
*   **Architecture**: Usually standard encoder/decoder blocks, similar functionally to Sparse AE.
*   **Loss Function Paradigm**: Custom inclusion mapping the `Frobenius norm of the Jacobian matrix` measuring the internal encoder activations measured relative against to the input parameters. Essentially, it forcefully punishes the algorithm for tracking enormous derivative trajectories.
*   *Note on Keras constraint*: Authentic contractive calculations demand mathematically rigid `tf.GradientTape()` setups measuring customized dynamic gradients. The conceptual code natively displays structural backbone utilizing a theoretical mock penalization setup.



In [ ]:
# Concept skeleton establishing baseline Contractive Autoencoder topological logic

input_img = Input(shape=(28,28,1))

x = Conv2D(16,(3,3),activation='relu',padding='same')(input_img)
encoded = MaxPooling2D((2,2),padding='same')(x)
# Initial layer compression

x = Conv2D(16,(3,3),activation='relu',padding='same')(encoded)
x = UpSampling2D((2,2))(x)
decoded = Conv2D(1,(3,3),activation='sigmoid',padding='same')(x)
# Decoding layout

# ================= CUSTOM CONTRACTIVE LOSS CONFIGURATION =================
import tensorflow.keras.backend as K

def contractive_loss(y_true, y_pred):
    # Retrieve base MSE reconstruction foundation tracking
    mse = K.mean(K.square(y_true - y_pred))
    
    # Mathematical Concept Implementation Reality Tracker:
    # Within full manual custom loop logic, calculate the dense Jacobian of the encapsulated hidden mapping
    # scaled fundamentally backward measuring precisely against source input structures.
    
    # Conceptual penalty function implementation:
    # return base_mse + (LAMBDA_STRENGTH_MODIFIER * || Mathematical Jacobian Map ||^2)
    
    return mse 
    # Directly functionally returning pseudo MSE currently mapping core structural logic

contractive_ae = Model(input_img, decoded)
contractive_ae.compile(optimizer='adam', loss=contractive_loss)
# Integrating topological customized tracker 

contractive_ae.fit(x_train, x_train, epochs=5, batch_size=128, validation_data=(x_test, x_test))

